In [1]:
import pandas as pd
import numpy as np

from tqdm import tqdm

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    classification_report
)

from imblearn.over_sampling import SMOTE

from xgboost import XGBClassifier
import os

os.environ["LOKY_MAX_CPU_COUNT"] = "8"

In [2]:
df = pd.read_csv(
    "Data/data_processed.csv"
)

In [3]:
X = df.drop(
    columns=["Class"]
)

y = df["Class"]


print(
    y.value_counts()
)

Class
0    283253
1       473
Name: count, dtype: int64


In [4]:
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)


print(
    "Train:",
    X_train.shape
)

print(
    "Valid:",
    X_valid.shape
)

print(
    "Test:",
    X_test.shape
)

Train: (226980, 30)
Valid: (28373, 30)
Test: (28373, 30)


In [5]:
smote = SMOTE(
    random_state=42,
    sampling_strategy=0.3
)


X_train_smote, y_train_smote = smote.fit_resample(
    X_train,
    y_train
)


print(
    y_train_smote.value_counts()
)

class_ratio = (
    y_train_smote.value_counts()[0]
    /
    y_train_smote.value_counts()[1]
)

Class
0    226602
1     67980
Name: count, dtype: int64


In [6]:
from xgboost import XGBClassifier


model = XGBClassifier(
    n_estimators=1000,

    learning_rate=0.20,

    max_depth=5,
    min_child_weight=5,

    subsample=0.779,
    colsample_bytree=0.893,

    gamma=1.48,

    reg_alpha=1.29,
    reg_lambda=1.18,

    eval_metric="aucpr",

    random_state=42,
    n_jobs=-1,

    scale_pos_weight=class_ratio,

    early_stopping_rounds=50
)

In [7]:
model.fit(
    X_train_smote,
    y_train_smote,

    eval_set=[
        (
            X_valid,
            y_valid
        )
    ],

    verbose=True
)

[0]	validation_0-aucpr:0.47747
[1]	validation_0-aucpr:0.53963
[2]	validation_0-aucpr:0.59749
[3]	validation_0-aucpr:0.63148
[4]	validation_0-aucpr:0.65896
[5]	validation_0-aucpr:0.64903
[6]	validation_0-aucpr:0.65862
[7]	validation_0-aucpr:0.65955
[8]	validation_0-aucpr:0.67523
[9]	validation_0-aucpr:0.67993
[10]	validation_0-aucpr:0.68271
[11]	validation_0-aucpr:0.68274
[12]	validation_0-aucpr:0.68190
[13]	validation_0-aucpr:0.68441
[14]	validation_0-aucpr:0.68314
[15]	validation_0-aucpr:0.68404
[16]	validation_0-aucpr:0.68492
[17]	validation_0-aucpr:0.68609
[18]	validation_0-aucpr:0.69924
[19]	validation_0-aucpr:0.70166
[20]	validation_0-aucpr:0.70223
[21]	validation_0-aucpr:0.70028
[22]	validation_0-aucpr:0.68540
[23]	validation_0-aucpr:0.68627
[24]	validation_0-aucpr:0.68750
[25]	validation_0-aucpr:0.72630
[26]	validation_0-aucpr:0.72109
[27]	validation_0-aucpr:0.71765
[28]	validation_0-aucpr:0.72306
[29]	validation_0-aucpr:0.73711
[30]	validation_0-aucpr:0.74617
[31]	validation_0-

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.893, device=None, early_stopping_rounds=50,
              enable_categorical=True, eval_metric='aucpr', feature_types=None,
              feature_weights=None, gamma=1.48, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.2, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
              max_leaves=None, min_child_weight=5, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=1000,
              n_jobs=-1, num_parallel_tree=None, ...)

In [8]:
y_pred = model.predict(
    X_test
)

y_prob = model.predict_proba(
    X_test
)[:,1]

In [9]:
from sklearn.metrics import classification_report


print(
    classification_report(
        y_test,
        y_pred,
        digits=4
    )
)

              precision    recall  f1-score   support

           0     0.9998    0.9996    0.9997     28326
           1     0.7692    0.8511    0.8081        47

    accuracy                         0.9993     28373
   macro avg     0.8845    0.9253    0.9039     28373
weighted avg     0.9994    0.9993    0.9993     28373

